# SAM-based Comb Detection

This notebook demonstrates using Segment Anything Model (SAM) to detect honeybee comb regions by identifying wooden frame boundaries.

## Approach for Two-Frame Images

Your images have **two wooden frames side by side**. SAM supports different prompting modes:

### 1. **Point Prompts (RECOMMENDED for your case)**
- Place 2 points: one in each frame (e.g., at 1/3 and 2/3 of width, centered vertically)
- SAM segments the objects at those locations
- Simple, fast, and effective for consistent frame layouts

### 2. **Automatic Mask Generation**
- SAM detects all objects automatically
- Select the largest masks (likely the frames)
- Good for exploration, slower

### 3. **Box Prompts**
- Provide bounding boxes around frames
- More precise but requires knowing exact boundaries

**Advantages of SAM approach:**
- Robust to different lighting conditions
- Doesn't require training on comb-specific data
- Good at detecting frame boundaries
- Can handle various comb types and conditions

**Requirements:**
```bash
pip install segment-anything
# Or from source:
pip install git+https://github.com/facebookresearch/segment-anything.git
```

**Download SAM checkpoint:**
- vit_h (best): https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
- vit_l: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth
- vit_b (fastest): https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

Save the checkpoint in your `models/` directory.

## ⚡ Quick Start (Recommended)

**For your two-frame images, follow these steps:**

1. **Setup** (cells 2-5): Install dependencies and configure paths
2. **Skip Method 1** (Automatic): Memory-intensive, not needed
3. **Use Method 2** (Point Prompts): ✅ Start here! Fast and efficient
4. **Batch Processing**: Process all images with point prompts
5. **Evaluation**: Compare with ground truth

**GPU Memory Requirements:**
- Point Prompts (Method 2): ~2-3 GB ✅ Most efficient
- Automatic (Method 1): ~8+ GB ⚠️ May cause errors
- Box Prompts (Method 3): ~2-3 GB ✅ Alternative

---

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from segmentation_restruct.comb_limitor import CombMaskGeneratorSAM

## Setup Paths

Configure paths to your SAM checkpoint and test images.

In [7]:
root_dir = Path().cwd().resolve().parent.parent
model_dir = root_dir / 'models'

# Path to SAM checkpoint (download from link above)
sam_checkpoint = model_dir / 'sam_vit_h_4b8939.pth'

# Verify checkpoint exists
if not sam_checkpoint.exists():
    print(f"⚠ SAM checkpoint not found: {sam_checkpoint}")
    print("\nPlease download from:")
    print("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth")
    print(f"\nSave to: {model_dir}")
else:
    print(f"✓ SAM checkpoint found: {sam_checkpoint.name}")

# Test image directory
images_dir = Path(r"E:\Bachelorarbeit\comb_limitation_dataset\images\test")

✓ SAM checkpoint found: sam_vit_h_4b8939.pth


## Method 1: Automatic Mask Generation (Optional - Memory Intensive)

**⚠️ WARNING**: This method is very memory-intensive and may cause out-of-memory errors!
- Requires ~8+ GB GPU memory for high-resolution images
- Much slower than point prompts
- **Skip this section if you have limited GPU memory**

**For your two-frame images, jump directly to Method 2 (Point Prompts)** which is:
- Much faster
- Uses less memory
- More accurate for consistent layouts

In [8]:
# Initialize SAM generator
generator = CombMaskGeneratorSAM(
    model_type="vit_h",                    # Use largest model for best accuracy
    checkpoint_path=str(sam_checkpoint),
    device="cuda",                         # Use "cpu" if no GPU
    points_per_side=32,                    # Higher = more detailed
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95
)

print("✓ SAM generator initialized (model will load on first use)")

✓ SAM generator initialized (model will load on first use)


### Reducing Memory Usage (if you need automatic detection)

If you must use automatic detection but have limited GPU memory:

1. **Resize the image** before processing:
   ```python
   # Resize to smaller resolution
   small_img = cv2.resize(image_rgb, (0, 0), fx=0.5, fy=0.5)
   ```

2. **Reduce `points_per_side`** (fewer points = less memory):
   ```python
   generator = CombMaskGeneratorSAM(
       points_per_side=16,  # Default: 32, try: 16 or 8
       ...
   )
   ```

3. **Use smaller model**:
   ```python
   generator = CombMaskGeneratorSAM(
       model_type="vit_b",  # Much smaller than vit_h
       ...
   )
   ```

4. **Use CPU instead of GPU** (slower but no memory limit):
   ```python
   generator = CombMaskGeneratorSAM(device="cpu", ...)
   ```

In [9]:
# Load a test image
test_images = list(images_dir.glob("*.png"))
if test_images:
    image_path = test_images[0]
    print(f"Loading: {image_path.name}")

    # Load image (SAM expects RGB)
    image = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    print(f"Image shape: {image_rgb.shape}")
else:
    print("⚠ No test images found")

Loading: background_cam-0_20250719T012105.601863.038Z.png
Image shape: (4608, 5312, 3)


In [ ]:
# ⚠️ SKIP THIS CELL IF YOU HAVE LIMITED GPU MEMORY ⚠️
# This method requires ~8+ GB GPU memory for large images
# Use Method 2 (Point Prompts) instead - much more efficient!

# Uncomment to run (only if you have sufficient GPU memory):
"""
# Generate comb mask using automatic detection
# This will take a moment on first run (loads SAM model)
comb_mask, masks_metadata = generator.generate_comb_mask_auto(
    image_rgb,
    select_largest=True,      # Select only the largest mask
    min_area_ratio=0.3        # Minimum 30% of image area
)

print(f"\nMask shape: {comb_mask.shape}")
print(f"Comb pixels: {np.sum(comb_mask):,}")
print(f"Coverage: {np.sum(comb_mask) / comb_mask.size * 100:.1f}%")
"""

print("⚠️ Automatic mask generation skipped (memory intensive)")
print("→ Proceed to Method 2 (Point Prompts) for efficient processing")


✓ SAM model loaded: vit_h
Generating masks with SAM (this may take a moment)...


OutOfMemoryError: CUDA out of memory. Tried to allocate 7.66 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 20.03 GiB is allocated by PyTorch, and 94.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Visualize results (only if you ran the automatic detection above)
"""
fig, axes = generator.visualize_detection(
    image_rgb,
    comb_mask,
    masks_metadata=masks_metadata,
    show_all_masks=True  # Show all detected masks
)
plt.show()
"""

print("⚠️ Visualization skipped - automatic detection not run")
print("→ See Method 2 for point-based detection and visualization")


## Method 2: Point-Prompted Detection (Recommended for Multiple Frames)

For images with **two frames side by side** (like yours), point prompts work best!
SAM will segment the objects at the specified point locations.

**Your case**: Two wooden frames → Use 2 points:
- Point 1: Center-left (h/2, w/3) 
- Point 2: Center-right (h/2, 2w/3)

In [ ]:
# Clear GPU memory (if needed after automatic detection or previous runs)
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✓ GPU memory cleared")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


In [ ]:
# Method 2a: Automatic point placement for 2 frames (default)
# This places points at height=50%, width=33% and 67%
comb_mask_points, metadata = generator.generate_comb_mask_points(
    image_rgb,
    height_ratio=0.5,      # Center vertically
    width_ratios=[0.33, 0.67]  # Two frames: 1/3 and 2/3 horizontally
)

print(f"\nMask coverage: {np.sum(comb_mask_points) / comb_mask_points.size * 100:.1f}%")
print(f"Score: {metadata['best_score']:.3f}")
print(f"Points used: {metadata['num_points']}")


In [ ]:
# Visualize point-prompted result
# The visualization will show the points on the input image!
fig, axes = generator.visualize_detection(
    image_rgb,
    comb_mask_points,
    masks_metadata=metadata  # Shows where points were placed
)
plt.show()


### Alternative Configurations

Adjust point positions based on your specific frame layout:

In [ ]:
# Example: 3 frames side by side
comb_mask_3frames, metadata_3 = generator.generate_comb_mask_points(
    image_rgb,
    height_ratio=0.5,
    width_ratios=[0.25, 0.5, 0.75]  # Three evenly spaced frames
)

# Example: Single frame centered
comb_mask_single, metadata_single = generator.generate_comb_mask_points(
    image_rgb,
    height_ratio=0.5,
    width_ratios=[0.5]  # One frame in the center
)

# Example: Custom manual points (in pixels)
h, w = image_rgb.shape[:2]
custom_points = np.array([
    [int(w * 0.3), int(h * 0.5)],   # Left frame
    [int(w * 0.7), int(h * 0.5)]    # Right frame
])
comb_mask_custom, metadata_custom = generator.generate_comb_mask_points(
    image_rgb,
    point_coords=custom_points
)

print(f"3 frames coverage: {np.sum(comb_mask_3frames) / comb_mask_3frames.size * 100:.1f}%")
print(f"Single frame coverage: {np.sum(comb_mask_single) / comb_mask_single.size * 100:.1f}%")
print(f"Custom points coverage: {np.sum(comb_mask_custom) / comb_mask_custom.size * 100:.1f}%")


In [ ]:
# Method 2b: Manual box specification
# Adjust coordinates based on your frame position
h, w = image_rgb.shape[:2]
custom_box = np.array([
    int(w * 0.1),   # x1 (left)
    int(h * 0.1),   # y1 (top)
    int(w * 0.9),   # x2 (right)
    int(h * 0.9)    # y2 (bottom)
])

comb_mask_custom, metadata_custom = generator.generate_comb_mask_box(
    image_rgb,
    box=custom_box
)

print(f"Custom box: {custom_box}")
print(f"Mask coverage: {np.sum(comb_mask_custom) / comb_mask_custom.size * 100:.1f}%")

## Method 3: Box-Prompted Detection (Alternative)

If you know the approximate frame boundaries, you can use box prompts.
However, for your case with two frames, **point prompts (Method 2) are recommended**.

## Batch Processing

Process multiple images using the best method for your data.

In [ ]:
from segmentation_restruct.comb_limitor import CombMaskEvaluator

# Initialize evaluator
evaluator = CombMaskEvaluator()

# Paths
gt_dir = Path(r"E:\Bachelorarbeit\comb_limitation_dataset\labels\test")
image_files = sorted(images_dir.glob("*.png"))

predicted_masks = []
ground_truth_masks = []
input_images = []
image_names = []

print(f"Processing {len(image_files)} images with SAM (point prompts)...")

for img_file in image_files:
    gt_file = gt_dir / img_file.name

    if not gt_file.exists():
        print(f"Skipping {img_file.name} - no ground truth")
        continue

    print(f"Processing {img_file.name}...")

    # Load image
    img = cv2.imread(str(img_file))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Generate mask with SAM using point prompts (best for 2 frames)
    pred_mask, _ = generator.generate_comb_mask_points(
        img_rgb,
        height_ratio=0.5,
        width_ratios=[0.33, 0.67]  # Two frames at 1/3 and 2/3
    )

    # Load ground truth
    gt_mask = cv2.imread(str(gt_file), cv2.IMREAD_GRAYSCALE)

    predicted_masks.append(pred_mask)
    ground_truth_masks.append(gt_mask)
    input_images.append(img_gray)
    image_names.append(img_file.name)

print(f"\n✓ Processed {len(predicted_masks)} images")


In [ ]:
# Evaluate batch
aggregate_metrics, per_image_metrics = evaluator.evaluate_batch_from_arrays(
    predicted_masks=predicted_masks,
    ground_truth_masks=ground_truth_masks,
    image_names=image_names
)

# Print results
evaluator.print_metrics(aggregate_metrics, title="SAM-based Comb Detection Results")

# Print per-image results
print("\n" + "="*60)
print("Per-Image Results:")
print("="*60)
for metrics in per_image_metrics:
    print(f"\n{metrics['filename']}:")
    print(f"  IoU: {metrics['iou']:.4f}  |  Dice: {metrics['dice']:.4f}  |  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}")

In [ ]:
# Visualize batch results
fig, axes = evaluator.visualize_batch(
    predicted_masks=predicted_masks,
    ground_truth_masks=ground_truth_masks,
    image_names=image_names,
    input_images=input_images,
    max_cols=2
)
plt.show()

In [ ]:
# Save results
from datetime import datetime

date_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_name = f"sam_vit_h_points_2frames_{date_str}"

saved_paths = evaluator.save_batch_results(
    aggregate_metrics=aggregate_metrics,
    per_image_metrics=per_image_metrics,
    output_dir=root_dir / "results",
    run_name=run_name,
    save_figure=True,
    predicted_masks=predicted_masks,
    ground_truth_masks=ground_truth_masks,
    input_images=input_images,
    image_names=image_names
)

print("\n✓ Results saved!")
for key, path in saved_paths.items():
    print(f"  {key}: {path.name}")


## Compare: SAM vs Segmentation-based Approach

Load and compare results from both approaches.

In [ ]:
# This cell is optional - compare SAM with your segmentation model results
# You would need to run the segmentation model first and load those results

print("Comparison:")
print(f"SAM approach - IoU: {aggregate_metrics['iou_mean']:.4f}")
print(f"SAM approach - Dice: {aggregate_metrics['dice_mean']:.4f}")
print("\nSAM is particularly good for:")
print("  • Robust frame detection")
print("  • Handling lighting variations")
print("  • No training data required")
print("\nSegmentation approach is better for:")
print("  • Cell-level detail")
print("  • Distinguishing cell types")
print("  • Fine-grained analysis")

## Tips for Best Results

**Method Selection (IMPORTANT):**
- **Point Prompts (Method 2)**: 🏆 RECOMMENDED
  - Fast (~1-2 seconds per image)
  - Low memory usage (~2-3 GB GPU)
  - Perfect for consistent frame layouts
  - What you should use for your two-frame images!
  
- **Automatic Detection (Method 1)**: ⚠️ Use with caution
  - Very slow (~10-30 seconds per image)
  - High memory usage (8+ GB GPU)
  - Good for exploration/debugging only
  - May cause out-of-memory errors

- **Box Prompts (Method 3)**: Alternative
  - Medium speed (~2-3 seconds)
  - Low memory usage
  - Good if frame positions vary

**Model Selection:**
- `vit_h`: Best accuracy, requires 6+ GB GPU (~2.5 GB for point prompts)
- `vit_l`: Good balance, requires 4+ GB GPU
- `vit_b`: Fastest, requires 2+ GB GPU, slightly less accurate

**Parameters to Tune (for point prompts):**
- `height_ratio`: Vertical position of points (0.5 = center)
- `width_ratios`: Horizontal positions for multiple frames
  - 2 frames: `[0.33, 0.67]` (default)
  - 3 frames: `[0.25, 0.5, 0.75]`
  - Custom spacing as needed

**Best Practices:**
1. ✅ **Start with point prompts** (Method 2) - fastest and most efficient
2. Clear GPU memory between experiments: `torch.cuda.empty_cache()`
3. For batch processing, use point prompts (much faster than automatic)
4. If you get out-of-memory errors, reduce image size or use CPU
5. Can combine SAM (frame detection) with segmentation (cell details)